# Stage 1 — Data Understanding and Initial Dataset Profiling

The first stage of this research project focuses on understanding the original UCI Bank Marketing dataset before applying any preprocessing, imbalance-handling technique, model training, or hyperparameter optimization. The dataset used in this study is `bank-full.csv`, which contains 45,211 customer records and 17 columns. The objective of this stage is to establish the structure, quality, target distribution, feature characteristics, and class imbalance of the raw dataset so that all subsequent experiments can be designed and evaluated systematically. First, the dataset was loaded using Pandas and its dimensions were verified, confirming that it contains 45,211 observations and 17 variables. The dataset columns were then identified to distinguish between predictor variables and the target variable. The input variables describe customer demographic information, financial characteristics, contact information, previous campaign interactions, and campaign-related attributes. The target variable is `y`, which represents whether the customer subscribed to a term deposit, with `yes` indicating subscription and `no` indicating non-subscription. The data types of all variables were then examined to identify numerical and categorical features. Numerical variables include attributes such as `age`, `balance`, `day`, `duration`, `campaign`, `pdays`, and `previous`, while categorical variables include `job`, `marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, and `poutcome`. The dataset was subsequently checked for actual missing values using Pandas' `isnull()` functionality. The raw dataset does not contain conventional missing values represented as `NaN`. However, several categorical variables contain the explicit value `unknown`. The `unknown` category was therefore analyzed separately rather than automatically treating it as a missing value. The analysis identified `288` unknown values in `job`, `1,857` in `education`, `13,020` in `contact`, and `36,959` in `poutcome`. These observations are retained in the dataset because they are valid recorded categories in the original data, and their treatment will be considered during the preprocessing and experimental stages rather than removing them arbitrarily. Duplicate records were also checked to determine whether repeated observations exist in the raw dataset. No data-cleaning operation is performed at this stage because the purpose of Stage 1 is to understand and document the original data before making preprocessing decisions. The target variable was then analyzed to determine the class distribution. The dataset contains 39,922 customers belonging to the `no` class and 5,289 customers belonging to the `yes` class. Therefore, approximately 88.3% of observations belong to the majority `no` class, while only 11.7% belong to the minority `yes` class. The majority-to-minority class ratio is approximately 7.55:1, confirming that the dataset represents an imbalanced binary classification problem. This imbalance is a central motivation for the proposed research because a model optimized only for overall accuracy could achieve a high accuracy by primarily predicting the majority class while performing poorly on the minority class that represents customers who subscribed to the term deposit. Therefore, later experiments will use imbalance-aware evaluation metrics and compare different imbalance-handling and hyperparameter-optimization strategies. Descriptive statistics were also examined for numerical variables to understand their distributions, ranges, central tendencies, and variability. Similarly, categorical variables were summarized using their unique-category counts and most frequent values. Finally, the number of unique values in every column was inspected to distinguish binary, low-cardinality categorical, and higher-cardinality numerical or categorical variables. No observations are removed during this initial data-understanding stage. The original dataset is preserved so that subsequent preprocessing, imbalance handling, hyperparameter optimization, threshold optimization, robustness analysis, and explainability experiments can be performed in a controlled and reproducible manner. Overall, Stage 1 establishes that the Bank Marketing dataset is a supervised binary classification dataset with mixed numerical and categorical predictors and a substantial class imbalance, making it appropriate for investigating imbalance-aware hyperparameter optimization and explainable machine learning.

In [14]:
import pandas as pd
import numpy as np

# Path to UCI Bank Marketing dataset
data_path = "../data/raw/bank-full.csv"
df = pd.read_csv(data_path)
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (45211, 17)


In [15]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,18,student,single,primary,no,1944,no,no,telephone,10,aug,122,3,-1,0,unknown,no
1,18,student,single,unknown,no,108,no,no,cellular,10,aug,167,1,-1,0,unknown,yes
2,18,student,single,primary,no,608,no,no,cellular,12,aug,267,1,-1,0,unknown,yes
3,18,student,single,unknown,no,35,no,no,telephone,21,aug,104,2,-1,0,unknown,no
4,18,student,single,secondary,no,5,no,no,cellular,24,aug,143,2,-1,0,unknown,no


In [16]:
print("Number of columns:", len(df.columns))
print("\nColumn names:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Number of columns: 17

Column names:

1. age
2. job
3. marital
4. education
5. default
6. balance
7. housing
8. loan
9. contact
10. day
11. month
12. duration
13. campaign
14. pdays
15. previous
16. poutcome
17. y


In [17]:
print("Data types of all columns:\n")
print(df.dtypes)

Data types of all columns:

age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object


In [18]:
# Check for missing values
missing_values = df.isnull().sum()

print("Missing values in each column:\n")
print(missing_values)

Missing values in each column:

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [19]:
# Total missing values in the complete dataset
print("\nTotal missing values:", df.isnull().sum().sum())


Total missing values: 0


In [20]:
# Count 'unknown' values in each column
unknown_counts = (df == "unknown").sum()

print("Number of 'unknown' values in each column:\n")
print(unknown_counts[unknown_counts > 0])

Number of 'unknown' values in each column:

job            288
education     1857
contact      13020
poutcome     36959
dtype: int64


In [21]:
print("Total actual missing values:", df.isnull().sum().sum())

Total actual missing values: 0


In [22]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [23]:
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Duplicate percentage:", round(duplicate_percentage, 2), "%")

Duplicate percentage: 0.0 %


In [24]:
# Analyze the target variable
target_counts = df["y"].value_counts()

print("Target distribution:")
print(target_counts)

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64


In [25]:
# Target distribution as percentages
target_percentages = df["y"].value_counts(normalize=True) * 100

print("\nTarget distribution (%):")
print(target_percentages.round(2))


Target distribution (%):
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


In [26]:
# Calculate class imbalance ratio

majority_count = df["y"].value_counts().max()
minority_count = df["y"].value_counts().min()

imbalance_ratio = majority_count / minority_count

print("Majority class count:", majority_count)
print("Minority class count:", minority_count)
print("Imbalance ratio:", round(imbalance_ratio, 2))

Majority class count: 39922
Minority class count: 5289
Imbalance ratio: 7.55


In [27]:
# Summary statistics for numerical features
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [28]:
# Summary statistics for categorical features
df.describe(include="object")

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


In [29]:
# Number of unique values in every column
unique_values = df.nunique().sort_values(ascending=False)

print("Unique values per column:\n")
print(unique_values)

Unique values per column:

balance      7168
duration     1573
pdays         559
age            77
campaign       48
previous       41
day            31
month          12
job            12
poutcome        4
education       4
marital         3
contact         3
loan            2
housing         2
default         2
y               2
dtype: int64
